In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import os
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
Q1_data_path = os.path.join(path, 'Q1_data.csv')
df_Q1_data = pd.read_csv(Q1_data_path)

In [ ]:
# Task 2: Write your code here:
df_Q1_data.head()

In [ ]:
# Task 3: Write your code here:
df_Q1_data.info()

In [ ]:
# Task 4: Write your code here:
df_Q1_data.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_Q1_data['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Delivery Time')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_Q1_data.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:

df_Q1_data[['Weather','Traffic_Level','Time_of_Day']]=df_Q1_data[['Weather','Traffic_Level','Time_of_Day']].fillna('none')
df_Q1_data[['Courier_Experience_yrs','Delivery_Time']] = df_Q1_data[['Courier_Experience_yrs','Delivery_Time']].fillna(df_Q1_data[['Courier_Experience_yrs','Delivery_Time']].mean())
df_Q1_data.isnull().sum()


In [ ]:
# Task 3: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df_Q1_data.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df_Q1_data.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")


In [ ]:
# Task 4: Write your code here:
le = LabelEncoder()
df_Q1_data['Weather'] = le.fit_transform(df_Q1_data['Weather'])
df_Q1_data['Traffic_Level'] = le.fit_transform(df_Q1_data['Traffic_Level'])
df_Q1_data['Time_of_Day'] = le.fit_transform(df_Q1_data['Time_of_Day'])
df_Q1_data['Vehicle_Type'] = le.fit_transform(df_Q1_data['Vehicle_Type'])
df_Q1_data.head()

In [ ]:
# Task 5: Write your code here:
X = df_Q1_data.drop('Delivery_Time', axis=1)
y = df_Q1_data['Delivery_Time']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head()

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = df_Q1_data.drop('Delivery_Time', axis=1)
y = df_Q1_data['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)

mae_scores = []

for train_idx, val_idx in kf.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)


print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
feature_cols = ['Distance_km'
,'Weather'
,'Traffic_Level'
,'Time_of_Day'
,'Vehicle_Type'
,'Preparation_Time_min'
,'Courier_Experience_yrs'
,'Delivery_Time']
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([min(y_test), max(y_test)],
         [min(y_test), max(y_test)],
         'r--', linewidth=2)

plt.xlabel("Actual y_test (Ground Truth)")
plt.ylabel("Predicted y_pred (Decision Tree)")
plt.title("Decision Tree Regression: Predictions vs. Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: